# Manuscript reproduction walkthrough

Run the repository analysis groups, inspect selected numerical results, and verify exported CSVs against fixed reference tables. Figures are rendered with the repository manuscript style and saved as PDFs. Coverage follows the reproduction driver. Several manuscript panels are still assembled separately.

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts" / "reproduce_manuscript.py").is_file()
     and (path / "styles" / "manuscript.mplstyle").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open the notebook from the repository root or its notebooks folder.")

output = root / "outputs" / "manuscript"
reference = root / "tests" / "reference_results"
groups = [
    "pareto", "model-validation", "interpretation", "experimental-validation",
    "extended-operation", "characterization", "transport",
]
reference_groups = {"pareto", "model-validation", "interpretation",
                    "extended-operation", "transport"}
use_tex = False  # Enable when the manuscript LaTeX dependencies are installed.

required = [root / "scripts" / "verify_results.py"]
if "transport" in groups:
    required.append(root / "data" / "literature" / "transport_reference_values.csv")
missing = [str(path.relative_to(root)) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Add required repository files: " + ", ".join(missing))

## Generate manuscript outputs

The model validation and SHAP groups take longer than the experimental summaries. Edit `groups` above to run a subset. The driver writes a separate subdirectory for each selected group.

In [2]:
import os

def run_script(name, *arguments):
    completed = subprocess.run(
        [sys.executable, str(root / "scripts" / name), *map(str, arguments)],
        cwd=root, capture_output=True, text=True, encoding="utf-8", errors="replace",
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    if completed.stdout:
        print(completed.stdout.rstrip())
    if completed.stderr:
        print(completed.stderr.rstrip(), file=sys.stderr)
    completed.check_returncode()

style_flags = [] if use_tex else ["--no-tex"]
run_script("reproduce_manuscript.py", "--output-dir", output, "--only", *groups, *style_flags)

Running pareto
Running model-validation
Validation dataset: 259 measurements, 112 conditions, 112 input groups.
Observed Pareto condition IDs: [82, 86, 89, 95]
Initial training window: 4 acquisition batches.

Acquisition-batch map:

Input groups containing multiple condition IDs:
Predicting batch 5/28: train on 16 earlier conditions (35 measurements); hold out 4 conditions
Predicting batch 6/28: train on 20 earlier conditions (45 measurements); hold out 4 conditions
Predicting batch 7/28: train on 24 earlier conditions (53 measurements); hold out 4 conditions
Predicting batch 8/28: train on 28 earlier conditions (61 measurements); hold out 4 conditions
Predicting batch 9/28: train on 32 earlier conditions (69 measurements); hold out 4 conditions
Predicting batch 10/28: train on 36 earlier conditions (78 measurements); hold out 4 conditions
Predicting batch 11/28: train on 40 earlier conditions (90 measurements); hold out 4 conditions
Predicting batch 12/28: train on 44 earlier conditio


ExactExplainer explainer: 100%|██████████| 112/112 [00:30<00:00,  4.90it/s]
ExactExplainer explainer: 113it [00:30,  4.13it/s]                         
ExactExplainer explainer: 113it [00:30,  3.44it/s]

ExactExplainer explainer: 100%|██████████| 112/112 [00:48<00:00,  2.18it/s]
ExactExplainer explainer: 113it [00:49,  1.64it/s]                         
ExactExplainer explainer: 113it [00:49,  1.53it/s]
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp


## Inspect reported results

The tables below show the held-out model errors, the five-hour ammonia balance, and the nitrogen-transport comparison when their groups are selected.

In [3]:
for group, table in [
    ("model-validation", "cv_metrics.csv"),
    ("extended-operation", "summary.csv"),
    ("transport", "ratio_summary.csv"),
]:
    if group in groups:
        print(table)
        display(pd.read_csv(output / group / table, float_precision="round_trip"))

for group in groups:
    for figure in sorted((output / group).glob("*.pdf")):
        print(figure.relative_to(root))

cv_metrics.csv


,Response,Evaluation unit,Number of conditions,RMSE,MAE,R2,RMSE and MAE unit
0,Independent rate GP,Condition mean,112,1.532954,0.990141,0.816333,nmol s^-1 cm^-2
1,FE-derived rate,Condition mean,112,1.347588,0.860199,0.858066,nmol s^-1 cm^-2
2,FE GP,Condition mean,112,4.146940,3.061788,0.776604,percentage points


summary.csv


,current_mA,charge_C,recovered_net_umol,endpoint_FE_percent,average_rate_nmol_s_cm2,endpoint_FE_SD_percent,average_rate_SD_nmol_s_cm2,water_blanks_shape_consistent,blank_delta_span_AU
0,8.1735,147.123,148.652085,29.246438,16.516898,0.464039,0.262065,True,0.000537


ratio_summary.csv


,Comparison salt (M),Reference salt (M),D ratio low,D ratio high,Central-solubility DC ratio low,Central-solubility DC ratio high,Full-scenario DC ratio low,Full-scenario DC ratio high
0,0.8,1.0,1.126832,1.136161,1.250783,1.261139,1.134344,1.613348


outputs\manuscript\pareto\S6.pdf
outputs\manuscript\interpretation\Fig4.pdf
outputs\manuscript\interpretation\S22.pdf
outputs\manuscript\experimental-validation\S12.pdf
outputs\manuscript\extended-operation\S13.pdf
outputs\manuscript\characterization\S27.pdf
outputs\manuscript\characterization\S28.pdf
outputs\manuscript\characterization\S29.pdf
outputs\manuscript\characterization\S30.pdf
outputs\manuscript\characterization\S31.pdf
outputs\manuscript\transport\S32.pdf


## Verify numerical outputs

The verifier compares parsed CSV values exactly against the fixed reference tables.

In [7]:
selected = [group for group in groups if group in reference_groups]
if selected:
    run_script("verify_results.py", "--output-dir", output,
               "--reference-dir", reference, "--only", *selected)
else:
    print("The selected groups have no numerical reference tables in this verifier.")

59 exact table matches, 0 failures.
